In [1]:
# Import required libraries
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

# Set publication-quality style
plt.rcParams['font.size'] = 14 * 1.2 
plt.rcParams['axes.labelsize'] = 16 * 1.2 
plt.rcParams['axes.titlesize'] = 18 * 1.2 
plt.rcParams['xtick.labelsize'] = 14 * 1.2 
plt.rcParams['ytick.labelsize'] = 14 * 1.2 
plt.rcParams['legend.fontsize'] = 12 * 1.2 

print("Libraries imported successfully")

Libraries imported successfully


## 1. Define Paths and Load Data

In [2]:
# Define base paths
base_path = Path('/home/renatob/data/FluoData1/aviris_dangermond/traits/datasets/clima_fit_prescribed_lai_ci')
pft_path = Path('/home/renatob/data/FluoData1/aviris_dangermond/California_Vegetation_WHRTYPE_Dangermond/output_latlon.nc')
output_dir = Path('/home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures')
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Base path: {base_path}")
print(f"Output directory: {output_dir}")

Base path: /home/renatob/data/FluoData1/aviris_dangermond/traits/datasets/clima_fit_prescribed_lai_ci
Output directory: /home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures


In [3]:
# Load trait-based maps
print("Loading trait-based (spatially explicit) maps...")
chl_trait_ds = xr.open_dataset(base_path / 'chl_aviris_dangermond_clima_fit_reg.nc')
lma_trait_ds = xr.open_dataset(base_path / 'lma_aviris_dangermond_clima_fit_reg.nc')

# Get coordinates
lats = chl_trait_ds['lat'].values
lons = chl_trait_ds['lon'].values

# Get time dimension info
n_times = len(chl_trait_ds['time'])
print(f"Number of time steps: {n_times}")
print(f"Map shape: {chl_trait_ds['chl'].shape}")
print(f"Lat range: {lats.min():.4f} to {lats.max():.4f}")
print(f"Lon range: {lons.min():.4f} to {lons.max():.4f}")

Loading trait-based (spatially explicit) maps...
Number of time steps: 13
Map shape: (13, 458, 492)
Lat range: 34.4413 to 34.5776
Lon range: -120.5019 to -120.3555


In [4]:
# Load PFT-based maps
print("\nLoading PFT-based (averaged) maps...")
chl_pft_ds = xr.open_dataset(base_path / 'mean_masked_chl_aviris_dangermond_clima_fit.nc')
lma_pft_ds = xr.open_dataset(base_path / 'mean_masked_lma_aviris_dangermond_clima_fit.nc')

# Load PFT map for masking
pft_ds = xr.open_dataset(pft_path)
pft_map = pft_ds['Band1'].values

# Create mask for valid PFTs (2, 3, 4)
pft_mask = np.isin(pft_map, [2, 3, 4])

print(f"PFT map shape: {pft_map.shape}")
print(f"Valid PFT pixels: {pft_mask.sum():,}")
print(f"Unique PFTs: {np.unique(pft_map[pft_mask])}")


Loading PFT-based (averaged) maps...
PFT map shape: (458, 492)
Valid PFT pixels: 105,227
Unique PFTs: [2. 3. 4.]


## 2. Extract Beginning and End Time Steps

In [5]:
# Select first and last time steps (beginning and end of campaign)
print("Extracting beginning and end time steps...")

# Chlorophyll - trait-based
chl_trait_begin = chl_trait_ds['chl'].isel(time=0).values  # µg/cm²
chl_trait_end = chl_trait_ds['chl'].isel(time=-1).values

# LMA - trait-based (convert g/cm² to g/m²)
lma_trait_begin = lma_trait_ds['lma'].isel(time=0).values * 1e4  # g/cm² → g/m²
lma_trait_end = lma_trait_ds['lma'].isel(time=-1).values * 1e4

# PFT-based (already time-averaged)
chl_pft = chl_pft_ds['chl'].values.squeeze()  # µg/cm²
lma_pft = lma_pft_ds['lma'].values.squeeze()  # Already in g/m²

print(f"\nChl trait begin: min={np.nanmin(chl_trait_begin):.2f}, max={np.nanmax(chl_trait_begin):.2f} µg/cm²")
print(f"Chl trait end:   min={np.nanmin(chl_trait_end):.2f}, max={np.nanmax(chl_trait_end):.2f} µg/cm²")
print(f"LMA trait begin: min={np.nanmin(lma_trait_begin):.2f}, max={np.nanmax(lma_trait_begin):.2f} g/m²")
print(f"LMA trait end:   min={np.nanmin(lma_trait_end):.2f}, max={np.nanmax(lma_trait_end):.2f} g/m²")

Extracting beginning and end time steps...

Chl trait begin: min=0.05, max=80.00 µg/cm²
Chl trait end:   min=0.05, max=80.00 µg/cm²
LMA trait begin: min=7.00, max=498.00 g/m²
LMA trait end:   min=7.00, max=498.00 g/m²


In [6]:
# Calculate differences and apply mask
print("Calculating differences...")

# Chlorophyll differences
chl_diff_begin = np.where(pft_mask, chl_trait_begin - chl_pft, np.nan)
chl_diff_end = np.where(pft_mask, chl_trait_end - chl_pft, np.nan)

# LMA differences
lma_diff_begin = np.where(pft_mask, lma_trait_begin - lma_pft, np.nan)
lma_diff_end = np.where(pft_mask, lma_trait_end - lma_pft, np.nan)

# Apply mask to trait and PFT data
chl_trait_begin_masked = np.where(pft_mask, chl_trait_begin, np.nan)
chl_trait_end_masked = np.where(pft_mask, chl_trait_end, np.nan)
chl_pft_masked = np.where(pft_mask, chl_pft, np.nan)

lma_trait_begin_masked = np.where(pft_mask, lma_trait_begin, np.nan)
lma_trait_end_masked = np.where(pft_mask, lma_trait_end, np.nan)
lma_pft_masked = np.where(pft_mask, lma_pft, np.nan)

print("\nDifference statistics:")
print(f"Chl diff begin: mean={np.nanmean(chl_diff_begin):.2f}, std={np.nanstd(chl_diff_begin):.2f}")
print(f"Chl diff end:   mean={np.nanmean(chl_diff_end):.2f}, std={np.nanstd(chl_diff_end):.2f}")
print(f"LMA diff begin: mean={np.nanmean(lma_diff_begin):.2f}, std={np.nanstd(lma_diff_begin):.2f}")
print(f"LMA diff end:   mean={np.nanmean(lma_diff_end):.2f}, std={np.nanstd(lma_diff_end):.2f}")

Calculating differences...

Difference statistics:
Chl diff begin: mean=16.09, std=23.04
Chl diff end:   mean=-11.24, std=16.12
LMA diff begin: mean=37.17, std=111.15
LMA diff end:   mean=-22.70, std=42.75


## 3. Define Plotting Functions

In [7]:
def add_density_inset(ax, data, config, position='lower right'):
    """
    Add a density distribution inset to an axes (matching Figure 5 style).
    
    Parameters:
    - ax: matplotlib axes to add inset to
    - data: array of difference values
    - config: trait configuration dictionary
    - position: position of inset
    """
    # Create inset axes - 28% size
    if position == 'lower right':
        inset_ax = inset_axes(ax, width="28%", height="28%", loc='lower right',
                             bbox_to_anchor=(0.02, 0.05, 1, 1), bbox_transform=ax.transAxes,
                             borderpad=1.0)
    else:
        inset_ax = inset_axes(ax, width="28%", height="28%", loc=position, borderpad=1.0)
    
    # Remove NaNs and zeros
    valid_data = data[np.isfinite(data) & (data != 0)]
    
    if len(valid_data) > 0:
        # Plot histogram
        inset_ax.hist(valid_data, bins=50, color='gray', alpha=0.7, 
                     edgecolor='black', linewidth=0.5, density=True)
        
        # Add vertical line at zero
        inset_ax.axvline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.8)
        
        # Add mean line
        mean_val = np.nanmean(valid_data)
        inset_ax.axvline(mean_val, color='blue', linestyle='--', linewidth=1.5, alpha=0.8)
        
        # Style the inset
        unit_label = config['unit']
        inset_ax.set_xlabel('Δ ' + unit_label, fontsize=7)
        inset_ax.set_ylabel('', fontsize=7)  # No "Density" label
        inset_ax.tick_params(labelsize=5)
        inset_ax.grid(True, alpha=0.3, linewidth=0.5)
    
    return inset_ax

print("Plotting functions defined")

Plotting functions defined


## 4. Create Figure S1 - Chlorophyll (Chl)

In [8]:
# Configuration for Chlorophyll
chl_config = {
    'label': 'Chlorophyll Content',
    'unit': 'µg/cm²',
    'cmap': 'YlGn',
    'cmap_diff': 'RdBu_r',
    'vmin': 0,
    'vmax': 80,
    'vmin_diff': -30,
    'vmax_diff': 30
}

# Create figure with 2 rows (beginning, end) and 3 columns (Trait, PFT, Difference)
fig, axes = plt.subplots(2, 3, figsize=(18, 12), dpi=150)

# Column titles
column_titles = ['Trait-based Approach', 'PFT-based Approach', 'Difference (Trait - PFT)']

# Row labels - BOTH ROWS GET LABELS
row_labels = ['Start 2022-02-24 (Day 0)', 'End 2022-05-29 (Day 366)']

# Panel labels
panel_labels = ['a', 'b', 'c', 'd', 'e', 'f']

# Create meshgrid for plotting
LON, LAT = np.meshgrid(lons, lats)

# Data for each row
row_data = [
    (chl_trait_begin_masked, chl_pft_masked, chl_diff_begin, row_labels[0]),
    (chl_trait_end_masked, chl_pft_masked, chl_diff_end, row_labels[1])
]

# Loop through rows
for row_idx, (trait_data, pft_data, diff_data, row_label) in enumerate(row_data):
    panel_idx = row_idx * 3
    
    # Column 1: Trait-based approach
    ax1 = axes[row_idx, 0]
    im1 = ax1.pcolormesh(LON, LAT, trait_data, cmap=chl_config['cmap'],
                         vmin=chl_config['vmin'], vmax=chl_config['vmax'],
                         shading='auto', rasterized=True)
    ax1.set_aspect('equal')
    
    # Add coastlines
    ax1.plot(lons[[0, -1, -1, 0, 0]], lats[[0, 0, -1, -1, 0]], 'k-', linewidth=1.5, alpha=0.7)
    
    # Panel label OUTSIDE upper left
    ax1.text(-0.05, 1.05, f'({panel_labels[panel_idx]})', transform=ax1.transAxes,
            fontsize=22, fontweight='bold', va='bottom', ha='right')
    
    # Statistics box
    trait_mean = np.nanmean(trait_data)
    trait_std = np.nanstd(trait_data)
    stats_text = f'Mean: {trait_mean:.2f}\nSTD: {trait_std:.2f}'
    ax1.text(0.98, 0.98, stats_text, transform=ax1.transAxes,
            fontsize=16, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
    
    # Lat/lon labels with fewer ticks
    ax1.tick_params(labelsize=12)
    ax1.locator_params(axis='x', nbins=4)
    ax1.locator_params(axis='y', nbins=4)
    
    # Colorbar - VERTICAL
    cbar1 = plt.colorbar(im1, ax=ax1, orientation='vertical', pad=0.02, fraction=0.046)
    cbar1.set_label(f"{chl_config['label']} ({chl_config['unit']})", fontsize=17, fontweight='bold')
    cbar1.ax.tick_params(labelsize=14)
    
    # Row label on left - SHOW ON BOTH ROWS (no conditional)
    ax1.set_ylabel(row_label, fontsize=16, fontweight='bold')
    
    # Column 2: PFT-based approach
    ax2 = axes[row_idx, 1]
    im2 = ax2.pcolormesh(LON, LAT, pft_data, cmap=chl_config['cmap'],
                         vmin=chl_config['vmin'], vmax=chl_config['vmax'],
                         shading='auto', rasterized=True)
    ax2.set_aspect('equal')
    
    # Add coastlines
    ax2.plot(lons[[0, -1, -1, 0, 0]], lats[[0, 0, -1, -1, 0]], 'k-', linewidth=1.5, alpha=0.7)
    
    # Panel label
    ax2.text(-0.05, 1.05, f'({panel_labels[panel_idx + 1]})', transform=ax2.transAxes,
            fontsize=22, fontweight='bold', va='bottom', ha='right')
    
    # Statistics box
    pft_mean = np.nanmean(pft_data)
    pft_std = np.nanstd(pft_data)
    stats_text = f'Mean: {pft_mean:.2f}\nSTD: {pft_std:.2f}'
    ax2.text(0.98, 0.98, stats_text, transform=ax2.transAxes,
            fontsize=16, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
    
    ax2.tick_params(labelsize=12)
    ax2.locator_params(axis='x', nbins=4)
    ax2.locator_params(axis='y', nbins=4)
    
    # Colorbar
    cbar2 = plt.colorbar(im2, ax=ax2, orientation='vertical', pad=0.02, fraction=0.046)
    cbar2.set_label(f"{chl_config['label']} ({chl_config['unit']})", fontsize=17, fontweight='bold')
    cbar2.ax.tick_params(labelsize=14)
    
    # Column 3: Difference with density inset
    ax3 = axes[row_idx, 2]
    im3 = ax3.pcolormesh(LON, LAT, diff_data, cmap=chl_config['cmap_diff'],
                         vmin=chl_config['vmin_diff'], vmax=chl_config['vmax_diff'],
                         shading='auto', rasterized=True)
    ax3.set_aspect('equal')
    
    # Add coastlines
    ax3.plot(lons[[0, -1, -1, 0, 0]], lats[[0, 0, -1, -1, 0]], 'k-', linewidth=1.5, alpha=0.7)
    
    # Panel label
    ax3.text(-0.05, 1.05, f'({panel_labels[panel_idx + 2]})', transform=ax3.transAxes,
            fontsize=22, fontweight='bold', va='bottom', ha='right')
    
    # Statistics box
    diff_mean = np.nanmean(diff_data)
    diff_std = np.nanstd(diff_data)
    stats_text = f'Mean: {diff_mean:.2f}\nSTD: {diff_std:.2f}'
    ax3.text(0.98, 0.98, stats_text, transform=ax3.transAxes,
            fontsize=16, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
    
    ax3.tick_params(labelsize=12)
    ax3.locator_params(axis='x', nbins=4)
    ax3.locator_params(axis='y', nbins=4)
    
    # Colorbar
    cbar3 = plt.colorbar(im3, ax=ax3, orientation='vertical', pad=0.02, fraction=0.046)
    cbar3.set_label(f"Δ {chl_config['label']} ({chl_config['unit']})", fontsize=17, fontweight='bold')
    cbar3.ax.tick_params(labelsize=14)
    
    # Add density inset
    add_density_inset(ax3, diff_data, chl_config, position='lower right')

# Add column titles at the top
for col_idx, col_title in enumerate(column_titles):
    axes[0, col_idx].set_title(col_title, fontsize=22, fontweight='bold', pad=15)

# Adjust layout
plt.tight_layout(rect=[0.02, 0, 1, 0.99])

# Save figure
output_path_s1 = output_dir / 'figure_S1_chl_temporal_revised.png'
plt.savefig(output_path_s1, dpi=300, facecolor='white')
print(f"\nFigure S1 saved to: {output_path_s1}")

plt.show()
plt.close(fig)

print("\n✅ Figure S1 (Chlorophyll) generation complete!")

/tmp/ipykernel_613854/2957010922.py:143: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.02, 0, 1, 0.99])



Figure S1 saved to: /home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures/figure_S1_chl_temporal_revised.png


AttributeError: 'NoneType' object has no attribute '_get_renderer'

<Figure size 2700x1800 with 14 Axes>


✅ Figure S1 (Chlorophyll) generation complete!


## 5. Create Figure S2 - Leaf Mass per Area (LMA)

In [9]:
# Configuration for LMA
lma_config = {
    'label': 'Leaf Mass per Area',
    'unit': 'g/m²',
    'cmap': 'YlOrBr',
    'cmap_diff': 'RdBu_r',
    'vmin': 0,
    'vmax': 200,
    'vmin_diff': -80,
    'vmax_diff': 80
}

# Create figure
fig, axes = plt.subplots(2, 3, figsize=(18, 12), dpi=150)

# Data for each row
row_data = [
    (lma_trait_begin_masked, lma_pft_masked, lma_diff_begin, row_labels[0]),
    (lma_trait_end_masked, lma_pft_masked, lma_diff_end, row_labels[1])
]

# Loop through rows
for row_idx, (trait_data, pft_data, diff_data, row_label) in enumerate(row_data):
    panel_idx = row_idx * 3
    
    # Column 1: Trait-based approach
    ax1 = axes[row_idx, 0]
    im1 = ax1.pcolormesh(LON, LAT, trait_data, cmap=lma_config['cmap'],
                         vmin=lma_config['vmin'], vmax=lma_config['vmax'],
                         shading='auto', rasterized=True)
    ax1.set_aspect('equal')
    ax1.plot(lons[[0, -1, -1, 0, 0]], lats[[0, 0, -1, -1, 0]], 'k-', linewidth=1.5, alpha=0.7)
    
    ax1.text(-0.05, 1.05, f'({panel_labels[panel_idx]})', transform=ax1.transAxes,
            fontsize=22, fontweight='bold', va='bottom', ha='right')
    
    trait_mean = np.nanmean(trait_data)
    trait_std = np.nanstd(trait_data)
    stats_text = f'Mean: {trait_mean:.2f}\nSTD: {trait_std:.2f}'
    ax1.text(0.98, 0.98, stats_text, transform=ax1.transAxes,
            fontsize=16, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
    
    ax1.tick_params(labelsize=12)
    ax1.locator_params(axis='x', nbins=4)
    ax1.locator_params(axis='y', nbins=4)
    
    cbar1 = plt.colorbar(im1, ax=ax1, orientation='vertical', pad=0.02, fraction=0.046)
    cbar1.set_label(f"{lma_config['label']} ({lma_config['unit']})", fontsize=17, fontweight='bold')
    cbar1.ax.tick_params(labelsize=14)
    
    # Row label on left - SHOW ON BOTH ROWS (no conditional)
    ax1.set_ylabel(row_label, fontsize=16, fontweight='bold')
    
    # Column 2: PFT-based approach
    ax2 = axes[row_idx, 1]
    im2 = ax2.pcolormesh(LON, LAT, pft_data, cmap=lma_config['cmap'],
                         vmin=lma_config['vmin'], vmax=lma_config['vmax'],
                         shading='auto', rasterized=True)
    ax2.set_aspect('equal')
    ax2.plot(lons[[0, -1, -1, 0, 0]], lats[[0, 0, -1, -1, 0]], 'k-', linewidth=1.5, alpha=0.7)
    
    ax2.text(-0.05, 1.05, f'({panel_labels[panel_idx + 1]})', transform=ax2.transAxes,
            fontsize=22, fontweight='bold', va='bottom', ha='right')
    
    pft_mean = np.nanmean(pft_data)
    pft_std = np.nanstd(pft_data)
    stats_text = f'Mean: {pft_mean:.2f}\nSTD: {pft_std:.2f}'
    ax2.text(0.98, 0.98, stats_text, transform=ax2.transAxes,
            fontsize=16, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
    
    ax2.tick_params(labelsize=12)
    ax2.locator_params(axis='x', nbins=4)
    ax2.locator_params(axis='y', nbins=4)
    
    cbar2 = plt.colorbar(im2, ax=ax2, orientation='vertical', pad=0.02, fraction=0.046)
    cbar2.set_label(f"{lma_config['label']} ({lma_config['unit']})", fontsize=17, fontweight='bold')
    cbar2.ax.tick_params(labelsize=14)
    
    # Column 3: Difference with density inset
    ax3 = axes[row_idx, 2]
    im3 = ax3.pcolormesh(LON, LAT, diff_data, cmap=lma_config['cmap_diff'],
                         vmin=lma_config['vmin_diff'], vmax=lma_config['vmax_diff'],
                         shading='auto', rasterized=True)
    ax3.set_aspect('equal')
    ax3.plot(lons[[0, -1, -1, 0, 0]], lats[[0, 0, -1, -1, 0]], 'k-', linewidth=1.5, alpha=0.7)
    
    ax3.text(-0.05, 1.05, f'({panel_labels[panel_idx + 2]})', transform=ax3.transAxes,
            fontsize=22, fontweight='bold', va='bottom', ha='right')
    
    diff_mean = np.nanmean(diff_data)
    diff_std = np.nanstd(diff_data)
    stats_text = f'Mean: {diff_mean:.2f}\nSTD: {diff_std:.2f}'
    ax3.text(0.98, 0.98, stats_text, transform=ax3.transAxes,
            fontsize=16, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
    
    ax3.tick_params(labelsize=12)
    ax3.locator_params(axis='x', nbins=4)
    ax3.locator_params(axis='y', nbins=4)
    
    cbar3 = plt.colorbar(im3, ax=ax3, orientation='vertical', pad=0.02, fraction=0.046)
    cbar3.set_label(f"Δ {lma_config['label']} ({lma_config['unit']})", fontsize=17, fontweight='bold')
    cbar3.ax.tick_params(labelsize=14)
    
    # Add density inset
    add_density_inset(ax3, diff_data, lma_config, position='lower right')

# Add column titles
for col_idx, col_title in enumerate(column_titles):
    axes[0, col_idx].set_title(col_title, fontsize=22, fontweight='bold', pad=15)

# Adjust layout
plt.tight_layout(rect=[0.02, 0, 1, 0.99])

# Save figure
output_path_s2 = output_dir / 'figure_S2_lma_temporal_revised.png'
plt.savefig(output_path_s2, dpi=300, facecolor='white')
print(f"\nFigure S2 saved to: {output_path_s2}")

plt.show()
plt.close(fig)

print("\n✅ Figure S2 (LMA) generation complete!")

/tmp/ipykernel_613854/1311763664.py:115: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.02, 0, 1, 0.99])



Figure S2 saved to: /home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures/figure_S2_lma_temporal_revised.png


AttributeError: 'NoneType' object has no attribute '_get_renderer'

<Figure size 2700x1800 with 14 Axes>


✅ Figure S2 (LMA) generation complete!


## 6. Export High-Resolution Versions (600 DPI)

In [10]:
print("Creating high-resolution versions (600 DPI)...\n")

# Figure S1 - High resolution
fig, axes = plt.subplots(2, 3, figsize=(18, 12), dpi=600)

row_data = [
    (chl_trait_begin_masked, chl_pft_masked, chl_diff_begin),
    (chl_trait_end_masked, chl_pft_masked, chl_diff_end)
]

for row_idx, (trait_data, pft_data, diff_data) in enumerate(row_data):
    panel_idx = row_idx * 3
    
    for col_idx, data in enumerate([trait_data, pft_data, diff_data]):
        ax = axes[row_idx, col_idx]
        
        if col_idx == 2:
            im = ax.pcolormesh(LON, LAT, data, cmap=chl_config['cmap_diff'],
                             vmin=chl_config['vmin_diff'], vmax=chl_config['vmax_diff'],
                             shading='auto', rasterized=True)
            cbar = plt.colorbar(im, ax=ax, orientation='vertical', pad=0.02, fraction=0.046)
            cbar.set_label(f"Δ {chl_config['label']} ({chl_config['unit']})", fontsize=17, fontweight='bold')
            add_density_inset(ax, data, chl_config, position='lower right')
        else:
            im = ax.pcolormesh(LON, LAT, data, cmap=chl_config['cmap'],
                             vmin=chl_config['vmin'], vmax=chl_config['vmax'],
                             shading='auto', rasterized=True)
            cbar = plt.colorbar(im, ax=ax, orientation='vertical', pad=0.02, fraction=0.046)
            cbar.set_label(f"{chl_config['label']} ({chl_config['unit']})", fontsize=17, fontweight='bold')
        
        ax.set_aspect('equal')
        ax.plot(lons[[0, -1, -1, 0, 0]], lats[[0, 0, -1, -1, 0]], 'k-', linewidth=1.5, alpha=0.7)
        ax.text(-0.05, 1.05, f'({panel_labels[panel_idx + col_idx]})', transform=ax.transAxes,
               fontsize=22, fontweight='bold', va='bottom', ha='right')
        
        mean_val = np.nanmean(data)
        std_val = np.nanstd(data)
        ax.text(0.98, 0.98, f'Mean: {mean_val:.2f}\nSTD: {std_val:.2f}', transform=ax.transAxes,
               fontsize=16, va='top', ha='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
        
        ax.tick_params(labelsize=12)
        ax.locator_params(axis='x', nbins=4)
        ax.locator_params(axis='y', nbins=4)
        cbar.ax.tick_params(labelsize=14)
        
        # Add row label on first column - BOTH ROWS
        if col_idx == 0:
            ax.set_ylabel(row_labels[row_idx], fontsize=16, fontweight='bold')

for col_idx, col_title in enumerate(column_titles):
    axes[0, col_idx].set_title(col_title, fontsize=22, fontweight='bold', pad=15)

plt.tight_layout(rect=[0.02, 0, 1, 0.99])
output_s1_hires = output_dir / 'figure_S1_chl_temporal_600dpi.png'
plt.savefig(output_s1_hires, dpi=600, facecolor='white')
print(f"✓ Figure S1 (600 DPI): {output_s1_hires}")
plt.close(fig)

# Figure S2 - High resolution
fig, axes = plt.subplots(2, 3, figsize=(18, 12), dpi=600)

row_data = [
    (lma_trait_begin_masked, lma_pft_masked, lma_diff_begin),
    (lma_trait_end_masked, lma_pft_masked, lma_diff_end)
]

for row_idx, (trait_data, pft_data, diff_data) in enumerate(row_data):
    panel_idx = row_idx * 3
    
    for col_idx, data in enumerate([trait_data, pft_data, diff_data]):
        ax = axes[row_idx, col_idx]
        
        if col_idx == 2:
            im = ax.pcolormesh(LON, LAT, data, cmap=lma_config['cmap_diff'],
                             vmin=lma_config['vmin_diff'], vmax=lma_config['vmax_diff'],
                             shading='auto', rasterized=True)
            cbar = plt.colorbar(im, ax=ax, orientation='vertical', pad=0.02, fraction=0.046)
            cbar.set_label(f"Δ {lma_config['label']} ({lma_config['unit']})", fontsize=17, fontweight='bold')
            add_density_inset(ax, data, lma_config, position='lower right')
        else:
            im = ax.pcolormesh(LON, LAT, data, cmap=lma_config['cmap'],
                             vmin=lma_config['vmin'], vmax=lma_config['vmax'],
                             shading='auto', rasterized=True)
            cbar = plt.colorbar(im, ax=ax, orientation='vertical', pad=0.02, fraction=0.046)
            cbar.set_label(f"{lma_config['label']} ({lma_config['unit']})", fontsize=17, fontweight='bold')
        
        ax.set_aspect('equal')
        ax.plot(lons[[0, -1, -1, 0, 0]], lats[[0, 0, -1, -1, 0]], 'k-', linewidth=1.5, alpha=0.7)
        ax.text(-0.05, 1.05, f'({panel_labels[panel_idx + col_idx]})', transform=ax.transAxes,
               fontsize=22, fontweight='bold', va='bottom', ha='right')
        
        mean_val = np.nanmean(data)
        std_val = np.nanstd(data)
        ax.text(0.98, 0.98, f'Mean: {mean_val:.2f}\nSTD: {std_val:.2f}', transform=ax.transAxes,
               fontsize=16, va='top', ha='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))
        
        ax.tick_params(labelsize=12)
        ax.locator_params(axis='x', nbins=4)
        ax.locator_params(axis='y', nbins=4)
        cbar.ax.tick_params(labelsize=14)
        
        # Add row label on first column - BOTH ROWS
        if col_idx == 0:
            ax.set_ylabel(row_labels[row_idx], fontsize=16, fontweight='bold')

for col_idx, col_title in enumerate(column_titles):
    axes[0, col_idx].set_title(col_title, fontsize=22, fontweight='bold', pad=15)

plt.tight_layout(rect=[0.02, 0, 1, 0.99])
output_s2_hires = output_dir / 'figure_S2_lma_temporal_600dpi.png'
plt.savefig(output_s2_hires, dpi=600, facecolor='white')
print(f"✓ Figure S2 (600 DPI): {output_s2_hires}")
plt.close(fig)

print("\n" + "="*80)
print("ALL FIGURES GENERATED SUCCESSFULLY!")
print("="*80)
print("\n✅ Figures created:")
print(f"  • Figure S1 (300 DPI): {output_path_s1}")
print(f"  • Figure S1 (600 DPI): {output_s1_hires}")
print(f"  • Figure S2 (300 DPI): {output_path_s2}")
print(f"  • Figure S2 (600 DPI): {output_s2_hires}")
print("\n✅ Key features:")
print("  • Date labels on BOTH top and bottom rows")
print("  • Increased label sizes (14-22pt)")
print("  • Column titles: Trait-based, PFT-based, Difference")
print("  • Panel labels (a-f) outside plots, no boxes")
print("  • Statistics boxes (Mean/STD) in all panels")
print("  • Density insets in difference column")
print("  • Coastlines for geographic context")

Creating high-resolution versions (600 DPI)...



/tmp/ipykernel_613854/4231232416.py:54: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.02, 0, 1, 0.99])


✓ Figure S1 (600 DPI): /home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures/figure_S1_chl_temporal_600dpi.png


/tmp/ipykernel_613854/4231232416.py:111: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0.02, 0, 1, 0.99])


✓ Figure S2 (600 DPI): /home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures/figure_S2_lma_temporal_600dpi.png

ALL FIGURES GENERATED SUCCESSFULLY!

✅ Figures created:
  • Figure S1 (300 DPI): /home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures/figure_S1_chl_temporal_revised.png
  • Figure S1 (600 DPI): /home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures/figure_S1_chl_temporal_600dpi.png
  • Figure S2 (300 DPI): /home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures/figure_S2_lma_temporal_revised.png
  • Figure S2 (600 DPI): /home/renatob/data/FluoData1/aviris_dangermond/shift_dangermond_trait/figures/figure_S2_lma_temporal_600dpi.png

✅ Key features:
  • Date labels on BOTH top and bottom rows
  • Increased label sizes (14-22pt)
  • Column titles: Trait-based, PFT-based, Difference
  • Panel labels (a-f) outside plots, no boxes
  • Statistics boxes (Mean/STD) in all panels
  • Density in